In [1]:
import numpy as np
import pandas as pd

In [2]:
house_candidate = pd.read_csv("house_candidate.csv")
house_state = pd.read_csv("house_state.csv")
governors_county_candidate = pd.read_csv("governors_county_candidate.csv")

##### Since house election results are by district, we wanted to see how each district performed, in regards to the margin of victory. 

Did specific party win with absolute competitiveness in the given district? How does the pattern look in each state as a whole? 

In [3]:
sum_values = house_candidate.groupby('district')['total_votes'].sum() # calculate the number of total votes within each district. 
sum_values.name = 'sum_votes'

house_candidate = house_candidate.merge(sum_values, how = 'left', on='district') # merge back to the house candidate df
house_candidate['percentage'] = house_candidate['total_votes']/house_candidate['sum_votes'] # calculate percentage of votes for each party. 


In [4]:
house_candidate = house_candidate.sort_values(by = "sum_votes", ascending=False)

# only keep top two candidates to calculate the margin of victory. 
candidates = house_candidate.groupby(["district"]).head(2)

# transpose the table into wide format, based on winning situation
candidates_wide = candidates.pivot_table(
    index='district',
    columns='won',
    values='percentage'
)

candidates_wide = candidates_wide.rename(columns={False: 'loser_pct', True: 'winner_pct'})
# filter for absolute win:
abs_win = candidates_wide.loc[candidates_wide['winner_pct']==1]
abs_win['loser_pct']  = 0 # set loser_pct = 0
# merge back to original df, using update
candidates_wide.update(abs_win)

/var/folders/gr/l32pp4856qd5rrnc4_q_kbn00000gn/T/ipykernel_78206/1981850886.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  abs_win['loser_pct']  = 0 # set loser_pct = 0


In [5]:
# calculate MOV
candidates_wide['MOV'] = candidates_wide['winner_pct'] - candidates_wide['loser_pct']

# competitiveness using pd.cut
candidates_wide['competitiveness'] = pd.cut(
    candidates_wide['MOV'],
    bins=[0, 0.10, 0.20, 1.0],
    labels=['Competitive', 'Likely', 'Safe']
)

candidates_wide.query("competitiveness == 'Competitive'")
#candidates_wide['state'] = candidates_wide['district'].str.split().str[0]
candidates_wide

won,loser_pct,winner_pct,MOV,competitiveness
district,,,,
Alabama’s 1st district,0.355712,0.644288,0.288575,Safe
Alabama’s 2nd district,0.346827,0.652227,0.305400,Safe
Alabama’s 3rd district,0.324850,0.675150,0.350299,Safe
Alabama’s 4th district,0.176963,0.823037,0.646074,Safe
Alabama’s 5th district,0.000000,1.000000,1.000000,Safe
...,...,...,...,...
Wisconsin’s 5th district,0.398567,0.601433,0.202866,Safe
Wisconsin’s 6th district,0.407204,0.592250,0.185046,Likely
Wisconsin’s 7th district,0.392346,0.607654,0.215307,Safe


In [6]:
candidates.merge(candidates_wide, how = 'left', on = 'district')
candidates.dropna() # drop NA rows

,district,candidate,party,total_votes,won,sum_votes,percentage
1206,United States’s 0th district,The Democratic candidate,DEM,77138263,True,152572318,0.505585
1208,United States’s 0th district,Others,OTH,2883083,False,152572318,0.018897
415,Montana at large,Matt Rosendale,REP,339169,True,601509,0.563864
416,Montana at large,Kathleen Williams,DEM,262340,False,601509,0.436136
1166,Colorado’s 2nd district,Thom Atkinson,LIB,13657,False,515663,0.026484
...,...,...,...,...,...,...,...
1210,California’s 21st district,TJ Cox,DEM,84406,False,170334,0.495532
888,Texas’s 33rd district,Marc Veasey,DEM,105317,True,157606,0.668230
889,Texas’s 33rd district,Fabian Vasquez,REP,39638,False,157606,0.251501
876,Texas’s 29th district,Phil Kurtz,LIB,2328,False,156473,0.014878


Do the top 5 district political party preferences align with their state overall outcome? 
What are each party's top 5 supporters? 


In [7]:
# for DEM
print(candidates.query("party == 'DEM'").sort_values(by="percentage", ascending=False).head(5)[['district', 'party', 'total_votes','percentage', 'won']])

                            district party  total_votes  percentage   won
1023          Alabama’s 7th district   DEM       225742    1.000000  True
772         Tennessee’s 5th district   DEM       252155    1.000000  True
623   North Carolina’s 12th district   DEM       341457    1.000000  True
510          New York’s 5th district   DEM       229125    0.993263  True
287     Massachusetts’s 3rd district   DEM       286896    0.977369  True


In [8]:
print(candidates.query("party == 'REP'").sort_values(by="percentage", ascending=False).head(5)[['district', 'party', 'total_votes','percentage', 'won']])

                     district party  total_votes  percentage   won
946   Virginia’s 9th district   REP       271851    1.000000  True
1020   Alabama’s 5th district   REP       253094    1.000000  True
1050  Arkansas’s 1st district   REP       237596    1.000000  True
7      Florida’s 2nd district   REP       305337    0.978647  True
1021   Alabama’s 6th district   REP       274160    0.971300  True


In [9]:
print(candidates.query("party == 'LIB'").sort_values(by="pdistrictercentage", ascending=False).head(5)[['district', 'party', 'total_votes','percentage', 'won']])

KeyError: 'pdistrictercentage'

In [ ]:
print(candidates.query("party == 'IND'").sort_values(by="percentage", ascending=False).head(5)[['district', 'party', 'total_votes','percentage', 'won']])

                         district party  total_votes  percentage    won
302  Massachusetts’s 8th district   IND        72060    0.186974  False
737   Rhode Island’s 1st district   IND        35457    0.159496  False
299  Massachusetts’s 7th district   IND        38675    0.125304  False
140       Illinois’s 7th district   IND        19355    0.062410  False
879         Texas’s 30th district   IND        10851    0.041030  False


The top 5 districts in support of the Democrats are from Tennessee, New Carolina, Alabama, New York, and Massacheusets. Would the election results for house candidates at district be indicative of their state election results for president in 2020?

In [ ]:
president_county = pd.read_csv("president_county_candidate.csv")

president_county.head()
sum_prez = president_county.groupby("state")['total_votes'].sum()
sum_prez.name = 'sum_votes'

president_county = president_county.merge(sum_prez, how = 'left', on='state') # merge back to the house candidate df
president_county['percentage'] = president_county['total_votes']/house_candidate['sum_votes'] # calculate percentage of votes for each party. 
president_county

,state,county,candidate,party,total_votes,won,sum_votes,percentage
0,Delaware,Kent County,Joe Biden,DEM,44552,True,504010,0.091245
1,Delaware,Kent County,Donald Trump,REP,41009,False,504010,0.083988
2,Delaware,Kent County,Jo Jorgensen,LIB,1044,False,504010,0.002138
3,Delaware,Kent County,Howie Hawkins,GRN,420,False,504010,0.000860
4,Delaware,New Castle County,Joe Biden,DEM,195034,True,504010,0.444712
...,...,...,...,...,...,...,...,...
32172,Arizona,Maricopa County,Write-ins,WRI,1331,False,3387326,NaN
32173,Arizona,Mohave County,Donald Trump,REP,78535,True,3387326,NaN
32174,Arizona,Mohave County,Joe Biden,DEM,24831,False,3387326,NaN
32175,Arizona,Mohave County,Jo Jorgensen,LIB,1302,False,3387326,NaN
